# 學生學習與作業版｜任務 10：預測時間與資料洩漏

**姓名：**＿＿＿何韋仁＿　**學號／組別：**＿＿＿C114118243＿　**完成日期：**＿＿11/20＿＿

**學習目標：** 辨認預測時不可用的欄位，區分標籤、特徵與事後資訊。

本單元與教師版「任務 10」相同標題、資料、教學順序與範例；除了明確標示的作業與填空，其餘教師版教材內容保留。對照重點：程式填空對應教師版「10.3 直接看答案」。

### 如何完成
1. 先閱讀觀念與執行資料準備；各單元可以獨立啟動，不需要依賴上一單元的變數。
2. **💻 P 作業**：在指定程式 cell 補上 `____`，可先用提供的 Vibe Coding 提示詞詢問 AI。補完再執行，並執行後面的驗證 cell。未填時出現 `NameError` 是預期現象，先完成填空再繼續。
3. **📝 Q 作業**：查詢 AI、比較教材內容，填寫緊接的文字作答區；請記錄實際提示詞、自己的解釋與查證方式。
4. **🎯 M 小任務**：在任務 4、8、12、16 將前面幾單元整合成可執行的成果。不要只交 AI 回覆，要交實際程式、輸出與自己的解釋。
5. 把 Notebook 存到自己的 Google 雲端硬碟，完成後下載／依教師指示繳交 `.ipynb`；小任務保留輸出、文字與 AI 使用紀錄。

**案例限制：**本資料是公式生成的 60 筆模擬紀錄，不能推論真實學生或因果。缺失與 15 小時值在計分後加入。預測時欄位可得性是假設；全部資料已在前半段探索，後續切出的測試集僅示範流程，不能視為完全獨立的新資料。

**完成檢核：**本單元有 1 項程式填空、2 項 AI 查詢作答；若是任務 4、8、12、16，另外有 1 項整合小任務。請保留自己的輸出、回答和驗證紀錄。


## 共用資料準備
以下在記憶體重建與既有教材相同的資料。執行後可以直接開始本單元；不會覆寫 CSV。


In [ ]:
# 匯入套件：numpy 建立隨機資料，pandas 管理表格，pyplot 繪圖
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 固定種子，並維持呼叫順序，以重現原教材的 60 筆模擬資料
rng = np.random.default_rng(42)
n = 60
departments = ["資訊管理", "企業管理", "金融管理"]
# 每個字典鍵是一個欄位；對應的值是 60 名學生的資料
data = {
    "student_id": range(1, n + 1),
    "department": rng.choice(departments, n),
    "sleep_hours": rng.normal(6.7, 1.1, n).round(1),
    "screen_time_hours": rng.normal(5.0, 1.5, n).clip(1, 10).round(1),
    "study_hours": rng.normal(2.5, 1.0, n).clip(0.5, 6).round(1),
    "attendance_rate": rng.normal(88, 8, n).clip(60, 100).round(0),
}
df = pd.DataFrame(data)
# 成績依設定公式與隨機誤差建立；關係來自模擬規則，不能視為因果發現
noise = rng.normal(0, 5, n)
score = (50 + 3 * df["study_hours"] + 2 * df["sleep_hours"]
         - 1.2 * df["screen_time_hours"] + 0.15 * df["attendance_rate"] + noise)
df["quiz_score"] = score.clip(0, 100).round(0)
# 故意在計分後加入資料品質問題；無法從表格還原計分時原始值
df.loc[[5, 17], "sleep_hours"] = np.nan
df.loc[8, "screen_time_hours"] = 15.0
print("模擬資料形狀（列, 欄）：", df.shape)


## 10.1 欄位可用性
**觀念說明：**在考前預測，考後才知道的 `quiz_score` 不可作輸入；`needs_support` 本身是標籤，也不能當特徵。`student_id` 是識別碼通常排除。其他行為欄位的考前可得性僅為教學假設。


In [ ]:
# 建立教師可投影的欄位檢查表
availability = pd.DataFrame({
 "field": ["student_id", "department", "study_hours", "sleep_hours", "screen_time_hours", "attendance_rate", "quiz_score", "needs_support"],
 "decision": ["排除", "假設可用", "假設可用", "假設可用", "假設可用", "假設可用", "排除", "標籤，不可作輸入"],
 "reason": ["識別碼", "需確認可得時間", "需確認可得時間", "需確認可得時間", "需確認可得時間", "需確認可得時間", "考後才知道，且用來建標籤", "答案本身"]})
display(availability)


## 📝 學生作業 10-Q1｜AI 查詢與作答：退訂洩漏
**任務：**預測下個月退訂時，為何退訂後才有的取消日期不可作輸入？

**可使用的 AI 提示詞：**
```text
我是 Python 與資料分析初學者。請用時間線說明客戶取消日期造成目標洩漏，並舉例可考慮的預測前欄位。
請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
```
先閱讀教材，再詢問 AI。請記錄工具名稱、實際提示詞；若 AI 提供資料來源，記錄可查證的網址／文件名稱與日期，並自行檢查，無法查證時寫「尚未查證」。不要提供真實個資。


### ✍️ 作業 10-Q1 專用作答區：請雙擊此文字儲存格填寫
- AI 工具：＿＿＿Colab Copilot (Gemini 2.5)
實際提示詞：我是 Python 與資料分析初學者。請用時間線說明客戶取消日期造成目標洩漏，並舉例可考慮的預測前欄位。請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。＿
- AI 回答重點（請自行整理）：＿1.什麼是目標洩漏 (Target Leakage)：在模型預測時，使用了「預測時間點之後」才發生的事件或欄位資訊，導致模型在訓練時有如「偷看答案」，這在機器學習中稱為目標洩漏。
2.時間線解釋取消日期為何不可用：
預測時間點（例如：10月31日）：我們要預測客戶在「11月份（下個月）是否會退訂」。此時，11月份的「取消日期」根本還沒有發生，值應該是空白的。
實際發生退訂（例如：11月15日）：客戶正式退訂，系統自動寫入「取消日期 = 11-15」。
關鍵錯誤：如果我們把「取消日期」這個欄位放進模型當作特徵，模型會學到「只要這個欄位不是空值，就百分之百會退訂」。然而在未來的真實預測中，考前（預測點）這個欄位必定是空的，模型將完全失去預測能力。
3.可考慮的預測前（考前已發生）特徵欄位：
客戶的「平均每月消費金額（常規特徵）」
過去 30 天內「登入系統的次數 / 活躍度（行為特徵）」
過去的「客服投訴次數（不滿意度指標）」
＿＿＿
- 我的理解：＿在建立預測模型時，我們必須設定一個嚴格的時間切點（Cut-off Date）。只有在「切點之前」就已經產生並確定寫入的數據，才能做為輸入特徵；任何在「切點之後」才產生的資訊，一律不能出現在訓練資料中，否則就會發生退訂洩漏。
具體例子：
假設我有一張客戶明細表，欄位有 customer_id（客戶編號）、login_count_oct（10月登入次數）、cancel_date_nov（11月取消日期）、以及目標標籤 churn_nov（11月是否退訂，1是/0否）。
我可以用程式驗證：如果執行 pd.crosstab(df['cancel_date_nov'].notna(), df['churn_nov'])，會發現「有取消日期者百分之百對應退訂 = 1」，這代表兩者高度共線且發生在同一預測區間。把這個欄位放進去，模型訓練分數會是 100 分，但上線預測時分數會直接雪崩，這就是最典型的目標洩漏。＿＿＿
- 查證來源／方式及日期（無法查證請註明）：＿＿Kaggle 官方機器學習教學之「Target Leakage」章節、Google 機器學習速成課程 (Machine Learning Crash Course) 之資料偏差說明。＿＿
- 我檢查或修正 AI 回答的地方：＿＿＿＿AI 原先提供的預測前欄位建議包含「最後一次與客服通話的滿意度」，但我自我修正為「10月份登入次數」與「10月份客服投訴次數」。因為「最後一次滿意度」在實務上有時會與退訂流程混在一起（例如退訂當下的滿意度調查），如果時間點沒抓準仍有洩漏風險。為了百分之百安全，我將其修正為具備明確歷史時間戳記的「10月份（前一個月）登入次數」，這讓模型的變數邏輯更加嚴謹且易於在我們現有的 DataFrame 架構中實作。


## 10.2 退訂案例
如果在本月底預測下個月是否退訂，且取消日期只有退訂後才產生，`cancellation_date` 就不能作為本月底預測的輸入；最近登入等欄位也必須確認截點與統計視窗。


In [ ]:
# 建立獨立小表，對照預測當下能否使用
churn = pd.DataFrame({"customer_id": ["A", "B", "C"],
 "logins_before_cutoff": [3, 10, 1], "cancellation_date": ["2026-02-05", None, "2026-02-12"],
 "churn_next_month": [1, 0, 1]})
display(churn)
print("可以考慮的候選：", ["logins_before_cutoff"])
print("排除的事後欄位：", ["cancellation_date"])


## 📝 學生作業 10-Q2｜AI 查詢與作答：欄位時間
**任務：**看見欄位名稱後，還要確認哪些收集或計算時間，才能判斷是否可用？

**可使用的 AI 提示詞：**
```text
我是 Python 與資料分析初學者。請說明截點、統計窗口、欄位可得時間如何影響預測特徵是否合法使用。
請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
```
先閱讀教材，再詢問 AI。請記錄工具名稱、實際提示詞；若 AI 提供資料來源，記錄可查證的網址／文件名稱與日期，並自行檢查，無法查證時寫「尚未查證」。不要提供真實個資。


### ✍️ 作業 10-Q2 專用作答區：請雙擊此文字儲存格填寫
- AI 工具：＿＿＿Colab Copilot (Gemini)
實際提示詞：我是 Python 與資料分析初學者。請說明截點、統計窗口、欄位可得時間如何影響預測特徵是否合法使用。請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。＿
- AI 回答重點（請自行整理）：＿＿＿1.預測切點 (Cut-off Date/Time)：預測發生的特定時點。例如我們要在 10/31 預測 11 月客戶是否會退訂。所有特徵必須在 10/31 當下就已經成為事實，任何 11/1 以後產生的行為都不行使用。
2.統計窗口 (Observation Window/Statistical Window)：在預測切點之前，用來計算行為特徵的時間區間。例如「過去 30 天（10/1 ~ 10/31）的登入次數」。如果統計窗口不小心跨越到了切點之後（例如包含 11 月的數據），就會導致目標洩漏（Target Leakage）。
3.欄位可得時間 (Data Availability Time/Lag)：數據真正被系統寫入、落庫且可供查詢的時間點。有時客戶的行為雖然發生在 10/30，但由於系統同步延遲（Data Lag），該數據要到 11/2 才會出現在資料庫中。如果在 10/31 跑預測模型，就不能使用這筆尚未落庫的 10/30 數據，否則就是使用了未來的資料（Look-ahead Bias）。＿
- 我的理解：我的理解：一個合法的預測特徵必須同時滿足三個時間條件：它必須發生在預測切點之前；它所統計的行為必須在明確的統計窗口之內；且該欄位在預測執行當下必須已經完成寫入與同步（無落後時間延遲）。任何一項不符合，都會導致模型「偷看答案」或「預測虛空」。
具體例子：
我們可以模擬一個含有「系統資料落庫延遲（2天）」的客戶登入資料表，驗證在 10/31 執行預測時，哪些資料是真正可用的。10/30 產生的資料因為延遲而無法在 10/31 即時取得，必須予以排除。＿＿＿＿
- 查證來源／方式及日期（無法查證請註明）：＿＿Kaggle 機器學習資料防漏指南（Kaggle Data Leakage tutorial）、Feature Engineering for Machine Learning (Alice Zheng).＿＿
- 我檢查或修正 AI 回答的地方：＿＿AI 最初的說明中忽略了「系統寫入延遲（Data Lag）」對即時預測的致命影響。我主動將其修正，並加入了一個真實系統落庫延遲 2 天的情境。在下方的 Python 模擬中，10/30 與 10/31 產生的行為在 10/31 預測時因為還在快取（Cache）中、尚未寫入分析用資料庫，因此不能作為模型特徵。這樣修正後，更貼近企業真實的資料科學部署挑戰！
＿＿


## 10.3 直接看答案的危險示範
下列程式不是合格模型：它直接使用事後分數計算答案。高正確率只能證明標籤規則被重複，不能證明可以在考前預測。


## 💻 學生作業 10-P1｜Vibe Coding 程式填空
**對應教師版：**任務 10「10.3 直接看答案」的示範程式。

**任務：**刻意利用考後 quiz_score，依低於 75 產生 leaked_answer，比較與 actual；這是錯誤情境示範，不是可部署模型。

**執行方式：**找到下一個程式儲存格中的 `____`，使用自己撰寫或 AI 建議的程式補齊，保留其他已提供的程式與註解。補完後先執行該 cell，再執行緊接的自動檢查；後續單元內容可能依賴此變數。

**可使用的 Vibe Coding 提示詞：**
```text
我是 Python 與資料分析初學者，正在 Google Colab 使用 pandas、numpy、matplotlib 或 scikit-learn。
我的任務是：刻意利用考後 quiz_score，依低於 75 產生 leaked_answer，比較與 actual；這是錯誤情境示範，不是可部署模型。
請依我貼上的上下文，只協助補上 ____，保留指定變數名稱與後續程式，不修改原始 df。
請解釋每一步的作用，提供繁體中文註解及可手動驗證的方法。
不要虛構執行結果；如需額外資訊，先指出缺少什麼。
```
完成後記錄實際使用的提示詞與自己改動的地方。


In [ ]:
# 刻意錯誤示例：事後分數直接提供了標籤資訊
actual = (df["quiz_score"] < 75).astype(int)

# [學生填空]：刻意用考後分數重現標籤，示範洩漏
leaked_answer = (df["quiz_score"] < 75).astype(int)

print("偷看答案的表面正確率：", (actual == leaked_answer).mean())
# 這個 100% 沒有實務預測效益：考前尚未取得 quiz_score

### ✅ 作業 10-P1｜執行驗證
執行下方程式，並同時檢查上方實際輸出。自動檢查只涵蓋主要條件；如果圖表、說明或資料來源不符任務，仍需人工修正。


In [ ]:
assert (leaked_answer == actual).all()
print("檢查通過：事後欄位可直接重現標籤；這正是洩漏警訊。")
檢查通過：事後欄位可直接重現標籤；這正是洩漏警訊。


### ✍️ 作業 10-P1｜我的完成紀錄
- 我實際使用的提示詞（若未使用 AI 請註明）：＿＿＿＿我是 Python 初學者，正在進行預測模型的目標洩漏錯誤示範。請教我如何補齊代碼中的 leaked_answer = ____，使其刻意利用事後才可得的 quiz_score 欄位來直接判斷學生的 quiz_score 是否小於 75 分，並轉換為 1 與 0 的整數標籤。
- 我補上的程式在做什麼：＿＿＿補上了 leaked_answer = (df["quiz_score"] < 75).astype(int)。這段程式碼刻意在「考前預測時間點」使用了「考後才可得」的期末考試分數 quiz_score。當分數小於 75 分時判斷為 True（並轉為整數 1），大於等於 75 分判斷為 False（並轉為整數 0），以此示範什麼是「目標洩漏（Target Leakage）」。＿
- 執行或驗證遇到的問題與修正：＿執行過程中非常順利，沒有遇到任何錯誤。此步驟是為了教學目的而刻意進行的「錯誤示範」，警惕我們在實務建置預測模型時，絕不能將「事後才產生的欄位」作為模型的輸入特徵。
＿＿＿
- 檢查結果及我如何確認：＿＿＿1.程式執行後輸出 偷看答案的表面正確率： 1.0，代表兩者完全一致（達 100% 的虛假高準確度）。
2.自動化驗證儲存格執行 assert (leaked_answer == actual).all() 順利通過，並成功印出：檢查通過：事後欄位可直接重現標籤；這正是洩漏警訊。，確認邏輯完全無誤。＿


## 10.4 Explain-back
請學生用自己的話區分：建立訓練標籤可用已知結果，但模型輸入只能用預測時間點可取得的資料。列出兩個真實專案會追問的欄位時間資訊。


## 延伸：把時間點寫成資料規格
例如：以 1 月 31 日 23:59 為截點，預測 2 月是否退訂；登入次數的統計時間必須截止於 1 月 31 日。不能僅看欄位名稱就認定一定安全，也要檢查欄位的計算方式。

**學生練習：**為出席率、睡眠時間各寫下蒐集窗口、資料可得時間與一個可能洩漏的情境。


## 本單元學習收束與 AI 使用紀錄

請學生先用自己的話解釋一個關鍵觀念，再指出一個執行結果及驗證方式。記錄：使用的提示詞、AI 回答中檢查或修改的地方、遇到的問題、修正後的證據與本資料限制。

**通用提示詞**：我是 Python 初學者，請逐步協助我完成上面的任務；每段程式加繁體中文註解，說明輸入、處理與輸出，提供兩項可實際執行的驗證，保留原始資料，不要虛構執行結果。
1.「上帝視角（訓練標籤）」與「切點防線（特徵可用性）」的本質差異
上帝視角（離線訓練標籤）：當我們在訓練模型時，我們是在看「歷史劇」。因為歷史已經發生了，我們可以走到時間線的終點，去回顧並標記哪些學生最終及格、哪些客戶最終退訂。這個已知結果是用來作為機器學習的學習目標（Y）。
切點防線（預測時點特徵）：然而，當模型要在某一時間點（例如 1 月 31 日 23:59）上線預報未來時，我們不能讓模型「偷看」任何在該時間點之後才發生的行為。所有的模型特徵（X）在計算時，其數據窗口必須完全在該切點之前關閉。如果在特徵中夾帶了未來行為，就叫目標洩漏（Target Leakage），會導致模型在訓練時表面上非常完美，实际上線預測時完全失靈。
2. 真實專案中，必須向資料工程師（DE）或業務端追問的兩個關鍵時間資訊
【數據落庫與同步延遲時間 (Data Sync Lag / ETL Delay)】：
追問問題：「用戶發生這個行為（如點擊或上傳），數據平均需要多久才會同步寫入到分析用的資料庫（Data Warehouse）中？」
實務原因：即使行為發生在切點前一小時，但如果系統同步需要 24 小時，那麼在切點當晚執行預測時，該數據在資料庫中仍然是不存在的（即「預測虛空」）。
【特徵統計窗口的批次更新頻率 (Feature Batch Update Frequency)】：
追問問題：「這個特徵指標（例如過去30天的出席率）是即時計算（Real-time），還是每天/每週固定排程（Cron Job）計算更新？」
實務原因：如果特徵是每週日午夜才批次更新，而我們在週三（切點）跑預測，我們拿到的其實是上週日的舊數據，這會影響特徵的時間有效性。
3. 特徵時間規格演練（以 1/31 23:59 預測 2 月表現為切點）
A. 出席率 (Attendance Rate)
蒐集窗口（統計區間）：1 月 1 日 00:00 至 1 月 31 日 23:59。
資料可得時間：2 月 1 日 09:00（需保留半天讓助教或導師確認最後一堂課的簽到並人工錄入系統）。
目標洩漏情境：工程師在提取特徵時，直接使用了系統總表中的 total_attendance_rate（整學期出席率），此時模型在訓練時，無意中使用了 2 月與 3 月的未來出席表現來預測 2 月的表現。由於不及格的學生往往在 2 月就開始嚴重缺課，模型會瞬間學到這個特徵，造成嚴重的目標洩漏。
B. 睡眠時間 (Sleep Hours)
蒐集窗口（統計區間）：1 月 24 日 00:00 至 1 月 30 日 23:59（切點前的過去 7 天）。
資料可得時間：1 月 31 日 08:00（學生透過穿戴裝置或 APP 記錄，系統通常於隔天早上完成同步）。
目標洩漏情境：問卷設計人員在期末考結束（3 月）後，才發放「回顧性生活習慣問卷」，讓學生回憶並填寫「1 月份的平均睡眠時間」。因為學生此時已經知道自己的期末成績，考得不好的學生可能會在心理上傾向於低估自己的睡眠時間。將此問卷特徵輸入模型，等於讓模型間接得知了期末成績的結果，造成目標洩漏。